# CrowdHumanでYOLO人物検出モデルをファインチューニング

CrowdHumanの **vbox（人物の見えている範囲）** を使い、`person` の1クラスの検出モデルを作成します。

現在はGPU確認とGoogle Drive接続だけを実装しています。以降の処理は段階ごとに追加します。

確認済みの保存場所:

- データ: `/content/drive/MyDrive/crowdhuman`
- 注釈: `raw/annotations/annotation_train.odgt`、`raw/annotations/annotation_val.odgt`（データフォルダからの相対パス）
- 保存済みデータの管理情報: `prepared/latest.json`（データフォルダからの相対パス）
- 管理情報の参照先: `prepared_9c497bb835b6435ea77e0992ab70b7d4.tar`（`prepared` 内）。設定は `vbox`。tarの中身は未確認です。

今後使用する保存先:

- 学習結果: `/content/drive/MyDrive/crowdhuman_vbox_ft/runs`
- 事前学習重み: `/content/drive/MyDrive/crowdhuman_vbox_ft/weights`
- ローカルデータ展開先: `/content/crowdhuman_vbox_yolo`


## 1. GPUを確認

Colabの「ランタイム」→「ランタイムのタイプを変更」でGPUを選択してから実行してください。
`nvidia-smi` の結果と、PyTorchから見えるGPU・メモリ・バージョンを表示します。


In [ ]:
import subprocess
import torch

try:
    result = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
except FileNotFoundError:
    print("nvidia-smiが見つかりません。ColabのGPU設定を確認してください。")

print(f"PyTorch: {torch.__version__}")
print(f"PyTorchのCUDAバージョン: {torch.version.cuda}")
print(f"CUDA利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {properties.name}")
        print(f"GPU {index}の総メモリ: {properties.total_memory / 1024**3:.2f} GiB")
else:
    print("GPUが利用できません。ランタイムのタイプを確認してください。")


## 2. Google Driveを接続

画面の案内に従って、CrowdHumanを保存しているGoogleアカウントのDriveを接続してください。


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
